# Notebook 09 — Ablación e interpretabilidad morfométrica

## Grain AI–MDPI

Este notebook estudia **qué información morfométrica utiliza el clasificador** y cómo se relaciona con las confusiones más importantes.

### Preguntas científicas

1. ¿Cuánto depende el campeón de información relacionada con el tamaño frente a la forma?
2. ¿Qué variables individuales muestran mayor importancia por permutación?
3. ¿Las conclusiones de importancia son estables entre calibración y prueba?
4. ¿Qué bloques morfométricos separan principalmente:
   - `SIRA` y `DERMASON`.
   - `INIAP-12` e `INIAP-20`?
5. ¿Cómo se relacionan estas observaciones con las diferencias inferenciales entre R1, R3 y R4 ya establecidas en el Notebook 06?

### Modelos congelados

- Dry Bean: `R1_full_native + rbf_svm`.
- INIAP Rice: `R4_hybrid_log_area + rbf_svm`.

### Métodos

- Auditoría de fidelidad del modelo guardado frente a las predicciones del Notebook 05.
- Una observación por `duplicate_group_id`.
- Importancia por permutación individual.
- Importancia por permutación conjunta de bloques.
- Sensibilidad por enmascaramiento con medianas de entrenamiento.
- Separación univariada y multivariada de pares de clases preespecificados.
- Reutilización de los contrastes pareados del Notebook 06.

### Restricciones

- No se reentrenan modelos.
- No se reajustan hiperparámetros.
- No se selecciona un nuevo campeón.
- No se utilizan resultados de prueba para modificar el modelo.
- La importancia por permutación es asociativa y dependiente del modelo; no es causal.
- El enmascaramiento con medianas puede crear combinaciones fuera de la distribución y se interpreta solo como sensibilidad.
- No se hacen afirmaciones sobre imágenes, cámaras, iluminación o segmentación.

In [ ]:
# ============================================================
# 1. Dependencias y entorno
# ============================================================
import importlib
import importlib.metadata
import subprocess
import sys

PINNED_SKLEARN = "1.6.1"

try:
    current_sklearn = importlib.metadata.version("scikit-learn")
except importlib.metadata.PackageNotFoundError:
    current_sklearn = None

packages = []

if current_sklearn != PINNED_SKLEARN:
    packages.append(f"scikit-learn=={PINNED_SKLEARN}")

for module_name, package_name in {
    "pyarrow": "pyarrow",
    "joblib": "joblib",
}.items():
    if importlib.util.find_spec(module_name) is None:
        packages.append(package_name)

if packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages]
    )

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import zipfile
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

print("Python:", sys.version.split()[0])
print("scikit-learn:", importlib.metadata.version("scikit-learn"))
print("Plataforma:", platform.platform())
print("CPU disponibles:", os.cpu_count())

In [ ]:
# ============================================================
# 2. Configuración reproducible
# ============================================================
RANDOM_SEED = 20260806
N_PERMUTATION_REPEATS = 200
PERMUTATION_INTERVAL = 0.95

DATASETS = ["dry_bean", "iniap_rice"]
PARTITIONS_FOR_IMPORTANCE = ["calibration", "test"]

PAIR_DEFINITIONS = {
    "dry_bean": ("SIRA", "DERMASON"),
    "iniap_rice": ("INIAP-12", "INIAP-20"),
}

FEATURE_BLOCKS = {
    "dry_bean": {
        "size_dependent": [
            "Area",
            "Perimeter",
            "MajorAxisLength",
            "MinorAxisLength",
            "ConvexArea",
            "EquivDiameter",
            "ShapeFactor1",
            "ShapeFactor2",
        ],
        "invariant_shape": [
            "AspectRation",
            "Eccentricity",
            "Extent",
            "Solidity",
            "roundness",
            "Compactness",
            "ShapeFactor3",
            "ShapeFactor4",
        ],
    },
    "iniap_rice": {
        "invariant_shape": [
            "aspect_ratio",
            "extent",
            "solidity",
            "circularity",
            "ellipse_fill",
        ],
        "size_anchor": [
            "log_area",
        ],
    },
}

LOCAL_ROOT = Path("/content/Grain_AI_MDPI_notebook09")
INPUT_DIR = LOCAL_ROOT / "00_input"
EXTRACT02_DIR = LOCAL_ROOT / "01_extracted_notebook02"
EXTRACT03_DIR = LOCAL_ROOT / "02_extracted_notebook03"
EXTRACT04_DIR = LOCAL_ROOT / "03_extracted_notebook04"
EXTRACT05_DIR = LOCAL_ROOT / "04_extracted_notebook05"
EXTRACT06_DIR = LOCAL_ROOT / "05_extracted_notebook06"
EXTRACT08_DIR = LOCAL_ROOT / "06_extracted_notebook08_v2"
OUTPUT_ROOT = LOCAL_ROOT / "07_outputs"

PROTOCOL_DIR = OUTPUT_ROOT / "03_protocol"
RESULTS_DIR = OUTPUT_ROOT / "07_results" / "09_ABLATION_INTERPRETABILITY"
FIGURES_DIR = OUTPUT_ROOT / "08_figures" / "09_ABLATION_INTERPRETABILITY"
TABLES_DIR = OUTPUT_ROOT / "09_tables" / "09_ABLATION_INTERPRETABILITY"

for directory in [
    INPUT_DIR,
    EXTRACT02_DIR,
    EXTRACT03_DIR,
    EXTRACT04_DIR,
    EXTRACT05_DIR,
    EXTRACT06_DIR,
    EXTRACT08_DIR,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Repeticiones de permutación:", N_PERMUTATION_REPEATS)
print("Bloques preespecificados:", FEATURE_BLOCKS)

In [ ]:
# ============================================================
# 3. Google Drive y localización de archivos
# ============================================================
try:
    from google.colab import drive, files
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None
    print("No se detectó Google Colab.")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Grain_Robustness_Q1")
DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

ZIP_FILENAMES = {
    "notebook02": "NOTEBOOK02_REDUNDANCY_REPRESENTATIONS_OUTPUTS.zip",
    "notebook03": "NOTEBOOK03_FROZEN_SPLITS_OUTPUTS.zip",
    "notebook04": "NOTEBOOK04_MODEL_SELECTION_OUTPUTS.zip",
    "notebook05": "NOTEBOOK05_BASELINE_CALIBRATION_OUTPUTS.zip",
    "notebook06": "NOTEBOOK06_PAIRED_INFERENCE_OUTPUTS.zip",
    "notebook08_v2": "NOTEBOOK08_V2_CANONICAL_CONFORMAL_OUTPUTS.zip",
}

def find_exact_file(root: Path, filename: str):
    if not root.exists():
        return None

    matches = sorted(root.rglob(filename))

    if not matches:
        return None

    return max(matches, key=lambda path: path.stat().st_mtime)

def obtain_input_zip(filename: str) -> Path:
    found = find_exact_file(DRIVE_PROJECT_ROOT, filename)

    if found is not None:
        print("Encontrado en Drive:", found)
        return found

    local_candidate = Path("/content") / filename

    if local_candidate.exists():
        print("Encontrado en /content:", local_candidate)
        return local_candidate

    if IN_COLAB and files is not None:
        print(f"No se encontró {filename}. Selecciónelo para subirlo.")
        uploaded = files.upload()

        if filename not in uploaded:
            raise FileNotFoundError(
                f"Debe subir exactamente {filename}."
            )

        destination = INPUT_DIR / filename
        destination.write_bytes(uploaded[filename])
        return destination

    raise FileNotFoundError(f"No se encontró {filename}.")

zip_paths = {
    key: obtain_input_zip(filename)
    for key, filename in ZIP_FILENAMES.items()
}

display(
    pd.DataFrame(
        [
            {"source": key, "path": str(path)}
            for key, path in zip_paths.items()
        ]
    )
)

In [ ]:
# ============================================================
# 4. Extracción y validaciones previas
# ============================================================
def clean_extract(zip_path: Path, destination: Path):
    if destination.exists():
        shutil.rmtree(destination)

    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(destination)

extract_map = {
    "notebook02": EXTRACT02_DIR,
    "notebook03": EXTRACT03_DIR,
    "notebook04": EXTRACT04_DIR,
    "notebook05": EXTRACT05_DIR,
    "notebook06": EXTRACT06_DIR,
    "notebook08_v2": EXTRACT08_DIR,
}

for key, destination in extract_map.items():
    clean_extract(zip_paths[key], destination)

SELECTED_CONFIG_FILE = (
    EXTRACT04_DIR
    / "07_results"
    / "04_MODEL_SELECTION"
    / "03_selected_model_configurations.csv"
)

REPRESENTATION_CATALOG_FILE = (
    EXTRACT02_DIR
    / "03_protocol"
    / "representation_catalog.csv"
)

NOTEBOOK06_VALIDATION_FILE = (
    EXTRACT06_DIR
    / "07_results"
    / "06_PAIRED_INFERENCE"
    / "02_final_inference_validation.csv"
)

REPRESENTATION_CONTRAST_FILE = (
    EXTRACT06_DIR
    / "09_tables"
    / "06_PAIRED_INFERENCE"
    / "Table_paired_representation_contrasts.csv"
)

MODEL_INTERVAL_FILE = (
    EXTRACT06_DIR
    / "09_tables"
    / "06_PAIRED_INFERENCE"
    / "Table_model_metric_intervals.csv"
)

NOTEBOOK08_VALIDATION_FILE = (
    EXTRACT08_DIR
    / "07_results"
    / "08_V2_CANONICAL_CONFORMAL"
    / "05_final_conformal_validation_v2.csv"
)

required_files = [
    SELECTED_CONFIG_FILE,
    REPRESENTATION_CATALOG_FILE,
    NOTEBOOK06_VALIDATION_FILE,
    REPRESENTATION_CONTRAST_FILE,
    MODEL_INTERVAL_FILE,
    NOTEBOOK08_VALIDATION_FILE,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Faltan archivos requeridos:\n" + "\n".join(missing_files)
    )

selected_configurations = pd.read_csv(SELECTED_CONFIG_FILE)
representation_catalog = pd.read_csv(REPRESENTATION_CATALOG_FILE)
notebook06_validation = pd.read_csv(NOTEBOOK06_VALIDATION_FILE)
representation_contrasts_all = pd.read_csv(REPRESENTATION_CONTRAST_FILE)
model_intervals_all = pd.read_csv(MODEL_INTERVAL_FILE)
notebook08_validation = pd.read_csv(NOTEBOOK08_VALIDATION_FILE)

if not notebook06_validation["passed"].astype(bool).all():
    display(notebook06_validation)
    raise RuntimeError("El Notebook 06 no superó sus validaciones.")

if not notebook08_validation["passed"].astype(bool).all():
    display(notebook08_validation)
    raise RuntimeError("El Notebook 08 v2 no superó sus validaciones.")

print("Notebooks 06 y 08 v2 validados.")

In [ ]:
# ============================================================
# 5. Carga de campeones, datos, modelos y predicciones
# ============================================================
training_champions = (
    selected_configurations.sort_values(
        ["dataset_id", "macro_f1_mean", "log_loss_mean", "candidate_id"],
        ascending=[True, False, True, True],
    )
    .groupby("dataset_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

if len(training_champions) != len(DATASETS):
    display(training_champions)
    raise RuntimeError("Número inesperado de campeones.")

def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

bundles = {}

for _, champion in training_champions.iterrows():
    dataset_id = champion["dataset_id"]
    representation = champion["representation"]
    algorithm = champion["algorithm"]

    split_path = (
        EXTRACT03_DIR
        / "05_frozen_splits"
        / dataset_id
        / f"{representation}_with_split.parquet"
    )

    model_path = (
        EXTRACT05_DIR
        / "05_models"
        / dataset_id
        / representation
        / f"{algorithm}.joblib"
    )

    metadata_path = (
        EXTRACT05_DIR
        / "05_models"
        / dataset_id
        / representation
        / f"{algorithm}_metadata.json"
    )

    if not split_path.exists():
        raise FileNotFoundError(split_path)
    if not model_path.exists():
        raise FileNotFoundError(model_path)
    if not metadata_path.exists():
        raise FileNotFoundError(metadata_path)

    frame = pd.read_parquet(split_path)
    model = joblib.load(model_path)
    metadata = load_json(metadata_path)

    feature_columns = metadata["feature_columns"]
    class_order = [str(label) for label in metadata["class_order"]]

    missing_features = sorted(set(feature_columns) - set(frame.columns))

    if missing_features:
        raise ValueError(
            f"Faltan variables en {dataset_id}: {missing_features}"
        )

    expected_block_features = set(
        feature
        for features in FEATURE_BLOCKS[dataset_id].values()
        for feature in features
    )

    if expected_block_features != set(feature_columns):
        raise RuntimeError(
            f"Los bloques no cubren exactamente las variables de {dataset_id}.\n"
            f"Bloques: {sorted(expected_block_features)}\n"
            f"Modelo: {sorted(feature_columns)}"
        )

    prediction_frames = {}

    for partition in ["calibration", "test"]:
        prediction_path = (
            EXTRACT05_DIR
            / "06_predictions"
            / "05_BASELINE"
            / dataset_id
            / representation
            / f"{algorithm}_{partition}_predictions.parquet"
        )

        if not prediction_path.exists():
            raise FileNotFoundError(prediction_path)

        prediction_frames[partition] = pd.read_parquet(prediction_path)

    bundles[dataset_id] = {
        "representation": representation,
        "algorithm": algorithm,
        "candidate_id": champion["candidate_id"],
        "frame": frame,
        "model": model,
        "metadata": metadata,
        "feature_columns": feature_columns,
        "class_order": class_order,
        "prediction_frames": prediction_frames,
    }

display(
    training_champions[
        [
            "dataset_id",
            "representation",
            "algorithm",
            "candidate_id",
            "macro_f1_mean",
            "macro_f1_sd",
        ]
    ]
)

In [ ]:
# ============================================================
# 6. Fidelidad de modelos y construcción de unidades de grupo
#    VERSIÓN CORREGIDA
# ============================================================
def macro_f1(y_true, y_pred, labels):
    return float(
        f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0,
        )
    )


def aligned_model_probabilities(model, X, class_order):
    """
    Reproduce la alineación de probabilidades del Notebook 05.
    """
    X = np.asarray(X, dtype=float)

    raw_probabilities = np.asarray(
        model.predict_proba(X),
        dtype=float,
    )

    estimator_classes = np.asarray(
        model.classes_,
        dtype=str,
    )

    class_order = [
        str(class_label)
        for class_label in class_order
    ]

    destination = {
        class_label: class_index
        for class_index, class_label
        in enumerate(class_order)
    }

    if set(estimator_classes) != set(class_order):
        raise RuntimeError(
            "Las clases del modelo y del metadata no coinciden."
        )

    aligned = np.zeros(
        (len(X), len(class_order)),
        dtype=float,
    )

    for source_index, class_label in enumerate(
        estimator_classes
    ):
        aligned[
            :,
            destination[class_label],
        ] = raw_probabilities[
            :,
            source_index,
        ]

    row_sums = aligned.sum(
        axis=1,
        keepdims=True,
    )

    if (
        not np.isfinite(aligned).all()
        or not np.isfinite(row_sums).all()
        or (row_sums <= 0).any()
    ):
        raise RuntimeError(
            "El modelo produjo probabilidades inválidas."
        )

    return aligned / row_sums


def labels_from_aligned_probabilities(
    probabilities,
    class_order,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    return np.asarray(
        class_order,
        dtype=str,
    )[probabilities.argmax(axis=1)]


fidelity_rows = []
group_audit_rows = []
group_frames = {}

for dataset_id, bundle in bundles.items():
    frame = bundle["frame"].copy()
    feature_columns = bundle["feature_columns"]
    class_order = bundle["class_order"]
    model = bundle["model"]

    for partition in [
        "train",
        "calibration",
        "test",
    ]:
        partition_frame = (
            frame.loc[
                frame["partition"] == partition
            ]
            .sort_values("record_id")
            .reset_index(drop=True)
        )

        label_nunique = (
            partition_frame.groupby(
                "duplicate_group_id"
            )["class_label"]
            .nunique()
        )

        feature_nunique = (
            partition_frame.groupby(
                "duplicate_group_id"
            )[feature_columns]
            .nunique(dropna=False)
            .max(axis=1)
        )

        if (label_nunique > 1).any():
            raise RuntimeError(
                "Un grupo contiene múltiples clases: "
                f"{dataset_id}, {partition}"
            )

        if (feature_nunique > 1).any():
            raise RuntimeError(
                "Un grupo contiene múltiples vectores: "
                f"{dataset_id}, {partition}"
            )

        group_frame = (
            partition_frame.groupby(
                "duplicate_group_id",
                as_index=False,
                sort=True,
            )
            .first()
        )

        group_frames[
            (dataset_id, partition)
        ] = group_frame

        group_audit_rows.append(
            {
                "dataset_id": dataset_id,
                "partition": partition,
                "rows": len(partition_frame),
                "groups": len(group_frame),
                "duplicate_rows_removed": (
                    len(partition_frame)
                    - len(group_frame)
                ),
                "groups_multiple_labels": int(
                    (label_nunique > 1).sum()
                ),
                "groups_multiple_vectors": int(
                    (feature_nunique > 1).sum()
                ),
            }
        )

    for partition in [
        "calibration",
        "test",
    ]:
        source_frame = (
            frame.loc[
                frame["partition"] == partition
            ]
            .copy()
        )

        source_frame["record_id"] = (
            source_frame["record_id"].astype(str)
        )

        source_frame = (
            source_frame.sort_values("record_id")
            .reset_index(drop=True)
        )

        prediction_frame = (
            bundle[
                "prediction_frames"
            ][partition]
            .copy()
        )

        prediction_frame["record_id"] = (
            prediction_frame["record_id"].astype(str)
        )

        prediction_frame = (
            prediction_frame.sort_values("record_id")
            .reset_index(drop=True)
        )

        if not source_frame["record_id"].equals(
            prediction_frame["record_id"]
        ):
            raise RuntimeError(
                "Desalineación de registros: "
                f"{dataset_id}, {partition}"
            )

        X_source = source_frame[
            feature_columns
        ].to_numpy(dtype=float)

        model_probabilities = (
            aligned_model_probabilities(
                model=model,
                X=X_source,
                class_order=class_order,
            )
        )

        probability_columns = [
            f"prob_native_{index}"
            for index in range(
                len(class_order)
            )
        ]

        saved_probabilities = (
            prediction_frame[
                probability_columns
            ].to_numpy(dtype=float)
        )

        absolute_differences = np.abs(
            model_probabilities
            - saved_probabilities
        )

        max_probability_difference = float(
            absolute_differences.max()
        )

        mean_probability_difference = float(
            absolute_differences.mean()
        )

        probability_match = bool(
            np.allclose(
                model_probabilities,
                saved_probabilities,
                rtol=0.0,
                atol=1e-10,
            )
        )

        predicted_labels = (
            labels_from_aligned_probabilities(
                model_probabilities,
                class_order,
            )
        )

        saved_labels = (
            prediction_frame[
                "predicted_native"
            ]
            .astype(str)
            .to_numpy()
        )

        label_match = bool(
            np.array_equal(
                predicted_labels,
                saved_labels,
            )
        )

        direct_model_labels = (
            model.predict(
                X_source
            ).astype(str)
        )

        direct_predict_difference_count = int(
            np.sum(
                direct_model_labels
                != predicted_labels
            )
        )

        fidelity_rows.append(
            {
                "dataset_id": dataset_id,
                "partition": partition,
                "rows": len(source_frame),
                "max_probability_difference": (
                    max_probability_difference
                ),
                "mean_probability_difference": (
                    mean_probability_difference
                ),
                "probabilities_match_at_1e_10": (
                    probability_match
                ),
                "probability_argmax_labels_match": (
                    label_match
                ),
                "direct_predict_vs_probability_argmax_differences": (
                    direct_predict_difference_count
                ),
                "passed": bool(
                    probability_match
                    and label_match
                ),
            }
        )


model_fidelity = pd.DataFrame(
    fidelity_rows
)

group_audit = pd.DataFrame(
    group_audit_rows
)

if not model_fidelity[
    "passed"
].astype(bool).all():
    display(model_fidelity)

    raise RuntimeError(
        "El modelo cargado no reproduce las "
        "probabilidades alineadas y las etiquetas "
        "por argmax del Notebook 05."
    )

model_fidelity.to_csv(
    RESULTS_DIR
    / "01_model_fidelity_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

group_audit.to_csv(
    RESULTS_DIR
    / "02_group_unit_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Fidelidad del modelo: CORRECTA")
display(model_fidelity)
display(group_audit)

## Importancia por permutación

Cada partición se reduce a una fila por grupo de duplicados. Para una variable individual, sus valores se permutan entre grupos. Para un bloque, todas las columnas del bloque se permutan usando el mismo orden, conservando su estructura interna.

La importancia se define como:

\[
\Delta F1_{\mathrm{macro}}
=
F1_{\mathrm{macro,base}}
-
F1_{\mathrm{macro,permutado}}
\]

Un valor positivo indica pérdida de desempeño después de romper la asociación entre la variable o bloque y la clase.

In [ ]:
# ============================================================
# 7. Funciones de importancia y sensibilidad
#    VERSIÓN CORREGIDA
# ============================================================
def probability_argmax_prediction(
    model,
    X,
    class_order,
):
    probabilities = (
        aligned_model_probabilities(
            model=model,
            X=X,
            class_order=class_order,
        )
    )

    return labels_from_aligned_probabilities(
        probabilities,
        class_order,
    )


def permutation_importance_block(
    model,
    X,
    y,
    labels,
    column_indices,
    n_repeats,
    seed,
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=str)

    baseline_prediction = (
        probability_argmax_prediction(
            model=model,
            X=X,
            class_order=labels,
        )
    )

    baseline_score = macro_f1(
        y,
        baseline_prediction,
        labels,
    )

    rng = np.random.default_rng(seed)
    drops = np.empty(
        n_repeats,
        dtype=float,
    )

    for repeat in range(n_repeats):
        permutation = rng.permutation(
            len(X)
        )

        X_permuted = X.copy()

        X_permuted[
            :,
            column_indices,
        ] = X[
            permutation
        ][
            :,
            column_indices,
        ]

        permuted_prediction = (
            probability_argmax_prediction(
                model=model,
                X=X_permuted,
                class_order=labels,
            )
        )

        drops[repeat] = (
            baseline_score
            - macro_f1(
                y,
                permuted_prediction,
                labels,
            )
        )

    alpha = (
        1.0
        - PERMUTATION_INTERVAL
    ) / 2.0

    return {
        "baseline_macro_f1": baseline_score,
        "importance_mean": float(
            drops.mean()
        ),
        "importance_sd": float(
            drops.std(ddof=1)
        ),
        "interval_low": float(
            np.quantile(
                drops,
                alpha,
            )
        ),
        "interval_high": float(
            np.quantile(
                drops,
                1.0 - alpha,
            )
        ),
        "minimum": float(
            drops.min()
        ),
        "maximum": float(
            drops.max()
        ),
        "n_repeats": int(
            n_repeats
        ),
    }


def masking_sensitivity(
    model,
    X,
    y,
    labels,
    column_indices,
    replacement_values,
):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=str)

    baseline_prediction = (
        probability_argmax_prediction(
            model=model,
            X=X,
            class_order=labels,
        )
    )

    baseline_score = macro_f1(
        y,
        baseline_prediction,
        labels,
    )

    X_masked = X.copy()

    X_masked[
        :,
        column_indices,
    ] = np.asarray(
        replacement_values,
        dtype=float,
    ).reshape(1, -1)

    masked_prediction = (
        probability_argmax_prediction(
            model=model,
            X=X_masked,
            class_order=labels,
        )
    )

    masked_score = macro_f1(
        y,
        masked_prediction,
        labels,
    )

    return {
        "baseline_macro_f1": baseline_score,
        "masked_macro_f1": masked_score,
        "macro_f1_drop": (
            baseline_score
            - masked_score
        ),
        "masked_accuracy": float(
            accuracy_score(
                y,
                masked_prediction,
            )
        ),
        "masked_mcc": float(
            matthews_corrcoef(
                y,
                masked_prediction,
            )
        ),
    }

In [ ]:
# ============================================================
# 8. Importancia individual y por bloques
# ============================================================
individual_rows = []
block_rows = []
masking_rows = []

for dataset_index, (dataset_id, bundle) in enumerate(bundles.items()):
    model = bundle["model"]
    feature_columns = bundle["feature_columns"]
    class_order = bundle["class_order"]

    train_group = group_frames[(dataset_id, "train")]
    train_medians = (
        train_group[feature_columns]
        .median(axis=0)
    )

    feature_to_index = {
        feature: index
        for index, feature in enumerate(feature_columns)
    }

    for partition_index, partition in enumerate(
        PARTITIONS_FOR_IMPORTANCE
    ):
        evaluation_frame = group_frames[
            (dataset_id, partition)
        ]

        X = evaluation_frame[
            feature_columns
        ].to_numpy(dtype=float)

        y = evaluation_frame[
            "class_label"
        ].astype(str).to_numpy()

        for feature_index, feature in enumerate(feature_columns):
            result = permutation_importance_block(
                model=model,
                X=X,
                y=y,
                labels=class_order,
                column_indices=[feature_index],
                n_repeats=N_PERMUTATION_REPEATS,
                seed=(
                    RANDOM_SEED
                    + dataset_index * 100000
                    + partition_index * 10000
                    + feature_index * 101
                ),
            )

            individual_rows.append(
                {
                    "dataset_id": dataset_id,
                    "representation": bundle["representation"],
                    "algorithm": bundle["algorithm"],
                    "partition": partition,
                    "feature": feature,
                    **result,
                }
            )

        for block_index, (block_name, block_features) in enumerate(
            FEATURE_BLOCKS[dataset_id].items()
        ):
            block_indices = [
                feature_to_index[feature]
                for feature in block_features
            ]

            result = permutation_importance_block(
                model=model,
                X=X,
                y=y,
                labels=class_order,
                column_indices=block_indices,
                n_repeats=N_PERMUTATION_REPEATS,
                seed=(
                    RANDOM_SEED
                    + dataset_index * 100000
                    + partition_index * 10000
                    + 5000
                    + block_index * 503
                ),
            )

            block_rows.append(
                {
                    "dataset_id": dataset_id,
                    "representation": bundle["representation"],
                    "algorithm": bundle["algorithm"],
                    "partition": partition,
                    "block": block_name,
                    "features": " | ".join(block_features),
                    "n_features": len(block_features),
                    **result,
                }
            )

            masking_result = masking_sensitivity(
                model=model,
                X=X,
                y=y,
                labels=class_order,
                column_indices=block_indices,
                replacement_values=train_medians[
                    block_features
                ].to_numpy(dtype=float),
            )

            masking_rows.append(
                {
                    "dataset_id": dataset_id,
                    "representation": bundle["representation"],
                    "algorithm": bundle["algorithm"],
                    "partition": partition,
                    "block": block_name,
                    "features": " | ".join(block_features),
                    "replacement_source": (
                        "training_group_level_feature_medians"
                    ),
                    **masking_result,
                }
            )

individual_importance = pd.DataFrame(individual_rows)
block_importance = pd.DataFrame(block_rows)
masking_sensitivity_results = pd.DataFrame(masking_rows)

individual_importance.to_csv(
    TABLES_DIR / "Table_individual_permutation_importance.csv",
    index=False,
    encoding="utf-8-sig",
)

block_importance.to_csv(
    TABLES_DIR / "Table_block_permutation_importance.csv",
    index=False,
    encoding="utf-8-sig",
)

masking_sensitivity_results.to_csv(
    TABLES_DIR / "Table_block_median_masking_sensitivity.csv",
    index=False,
    encoding="utf-8-sig",
)

display(block_importance)
display(masking_sensitivity_results)

In [ ]:
# ============================================================
# 9. Estabilidad calibración–prueba
# ============================================================
stability_rows = []

for dataset_id in DATASETS:
    calibration_table = (
        individual_importance.loc[
            (
                individual_importance["dataset_id"] == dataset_id
            )
            & (
                individual_importance["partition"] == "calibration"
            ),
            ["feature", "importance_mean"],
        ]
        .rename(
            columns={
                "importance_mean": "calibration_importance"
            }
        )
    )

    test_table = (
        individual_importance.loc[
            (
                individual_importance["dataset_id"] == dataset_id
            )
            & (
                individual_importance["partition"] == "test"
            ),
            ["feature", "importance_mean"],
        ]
        .rename(
            columns={
                "importance_mean": "test_importance"
            }
        )
    )

    merged = calibration_table.merge(
        test_table,
        on="feature",
        how="inner",
        validate="one_to_one",
    )

    correlation, p_value = spearmanr(
        merged["calibration_importance"],
        merged["test_importance"],
    )

    merged["dataset_id"] = dataset_id
    merged["absolute_difference"] = np.abs(
        merged["calibration_importance"]
        - merged["test_importance"]
    )

    stability_rows.append(
        {
            "dataset_id": dataset_id,
            "n_features": len(merged),
            "spearman_rho": float(correlation),
            "spearman_p_value_descriptive": float(p_value),
            "mean_absolute_importance_difference": float(
                merged["absolute_difference"].mean()
            ),
        }
    )

    merged.to_csv(
        RESULTS_DIR
        / f"03_feature_importance_stability_{dataset_id}.csv",
        index=False,
        encoding="utf-8-sig",
    )

importance_stability = pd.DataFrame(stability_rows)

importance_stability.to_csv(
    TABLES_DIR / "Table_importance_calibration_test_stability.csv",
    index=False,
    encoding="utf-8-sig",
)

display(importance_stability)

## Análisis dirigido de confusiones

Los pares de clases fueron definidos antes de este notebook a partir del patrón de errores de los campeones:

- Dry Bean: `SIRA` frente a `DERMASON`.
- INIAP Rice: `INIAP-12` frente a `INIAP-20`.

La separación univariada se calcula en el conjunto de desarrollo (`train + calibration`) usando:

- Diferencia de medias estandarizada.
- AUC univariada corregida por dirección: \(\max(AUC, 1-AUC)\).

La prueba se utiliza únicamente para describir errores y distancias a centroides aprendidos en entrenamiento.

In [ ]:
# ============================================================
# 10. Separación de pares y diagnóstico de errores
# ============================================================
pair_feature_rows = []
pair_block_rows = []
pair_confusion_rows = []
pair_sample_rows = []

for dataset_id, bundle in bundles.items():
    class_a, class_b = PAIR_DEFINITIONS[dataset_id]
    feature_columns = bundle["feature_columns"]
    class_order = bundle["class_order"]
    model = bundle["model"]

    train_frame = group_frames[(dataset_id, "train")].copy()
    calibration_frame = group_frames[
        (dataset_id, "calibration")
    ].copy()
    test_frame = group_frames[(dataset_id, "test")].copy()

    development_frame = pd.concat(
        [train_frame, calibration_frame],
        ignore_index=True,
    )

    pair_development = development_frame.loc[
        development_frame["class_label"].isin(
            [class_a, class_b]
        )
    ].copy()

    train_means = train_frame[feature_columns].mean(axis=0)
    train_sds = (
        train_frame[feature_columns]
        .std(axis=0, ddof=1)
        .replace(0.0, 1.0)
    )

    standardized_development = (
        pair_development[feature_columns]
        - train_means
    ) / train_sds

    binary_target = (
        pair_development["class_label"].astype(str)
        == class_a
    ).astype(int).to_numpy()

    for feature in feature_columns:
        values = standardized_development[
            feature
        ].to_numpy(dtype=float)

        mean_a = float(
            values[
                pair_development["class_label"].astype(str).to_numpy()
                == class_a
            ].mean()
        )

        mean_b = float(
            values[
                pair_development["class_label"].astype(str).to_numpy()
                == class_b
            ].mean()
        )

        raw_auc = float(
            roc_auc_score(binary_target, values)
        )

        pair_feature_rows.append(
            {
                "dataset_id": dataset_id,
                "class_a": class_a,
                "class_b": class_b,
                "feature": feature,
                "standardized_mean_class_a": mean_a,
                "standardized_mean_class_b": mean_b,
                "standardized_mean_difference_a_minus_b": (
                    mean_a - mean_b
                ),
                "absolute_standardized_mean_difference": abs(
                    mean_a - mean_b
                ),
                "univariate_auc_raw": raw_auc,
                "univariate_auc_direction_adjusted": max(
                    raw_auc,
                    1.0 - raw_auc,
                ),
                "development_groups": len(pair_development),
            }
        )

    for block_name, block_features in FEATURE_BLOCKS[
        dataset_id
    ].items():
        block_indices = [
            feature_columns.index(feature)
            for feature in block_features
        ]

        standardized_matrix = standardized_development[
            block_features
        ].to_numpy(dtype=float)

        mask_a = (
            pair_development["class_label"].astype(str).to_numpy()
            == class_a
        )

        mask_b = (
            pair_development["class_label"].astype(str).to_numpy()
            == class_b
        )

        centroid_a = standardized_matrix[mask_a].mean(axis=0)
        centroid_b = standardized_matrix[mask_b].mean(axis=0)

        pair_block_rows.append(
            {
                "dataset_id": dataset_id,
                "class_a": class_a,
                "class_b": class_b,
                "block": block_name,
                "features": " | ".join(block_features),
                "euclidean_centroid_distance": float(
                    np.linalg.norm(centroid_a - centroid_b)
                ),
                "mean_absolute_standardized_difference": float(
                    np.mean(np.abs(centroid_a - centroid_b))
                ),
            }
        )

    X_train_scaled = model.named_steps["scaler"].transform(
        train_frame[feature_columns].to_numpy(dtype=float)
    )

    train_labels = train_frame[
        "class_label"
    ].astype(str).to_numpy()

    centroids = {
        class_label: X_train_scaled[
            train_labels == class_label
        ].mean(axis=0)
        for class_label in [class_a, class_b]
    }

    pair_test = test_frame.loc[
        test_frame["class_label"].isin(
            [class_a, class_b]
        )
    ].copy()

    X_pair_test = pair_test[
        feature_columns
    ].to_numpy(dtype=float)

    pair_probabilities = (
        aligned_model_probabilities(
            model=model,
            X=X_pair_test,
            class_order=class_order,
        )
    )

    pair_predictions = (
        labels_from_aligned_probabilities(
            pair_probabilities,
            class_order,
        )
    )

    pair_scaled = model.named_steps[
        "scaler"
    ].transform(
        X_pair_test
    )

    class_to_probability_index = {
        str(label): index
        for index, label in enumerate(
            class_order
        )
    }

    true_labels = pair_test[
        "class_label"
    ].astype(str).to_numpy()

    for true_class in [class_a, class_b]:
        true_mask = true_labels == true_class

        for predicted_class in class_order:
            pair_confusion_rows.append(
                {
                    "dataset_id": dataset_id,
                    "class_a": class_a,
                    "class_b": class_b,
                    "true_class": true_class,
                    "predicted_class": predicted_class,
                    "count": int(
                        np.sum(
                            true_mask
                            & (
                                pair_predictions
                                == predicted_class
                            )
                        )
                    ),
                }
            )

    for row_index in range(len(pair_test)):
        true_class = true_labels[row_index]
        predicted_class = pair_predictions[row_index]

        probability_a = float(
            pair_probabilities[
                row_index,
                class_to_probability_index[class_a],
            ]
        )

        probability_b = float(
            pair_probabilities[
                row_index,
                class_to_probability_index[class_b],
            ]
        )

        pair_sample_rows.append(
            {
                "dataset_id": dataset_id,
                "record_id": str(
                    pair_test.iloc[row_index]["record_id"]
                ),
                "duplicate_group_id": str(
                    pair_test.iloc[row_index]["duplicate_group_id"]
                ),
                "true_class": true_class,
                "predicted_class": predicted_class,
                "correct": bool(true_class == predicted_class),
                "pair_to_pair_confusion": bool(
                    (
                        true_class == class_a
                        and predicted_class == class_b
                    )
                    or (
                        true_class == class_b
                        and predicted_class == class_a
                    )
                ),
                f"probability_{class_a}": probability_a,
                f"probability_{class_b}": probability_b,
                "pair_probability_margin_absolute": abs(
                    probability_a - probability_b
                ),
                f"distance_to_{class_a}_centroid": float(
                    np.linalg.norm(
                        pair_scaled[row_index]
                        - centroids[class_a]
                    )
                ),
                f"distance_to_{class_b}_centroid": float(
                    np.linalg.norm(
                        pair_scaled[row_index]
                        - centroids[class_b]
                    )
                ),
            }
        )

pair_feature_separation = pd.DataFrame(pair_feature_rows)
pair_block_separation = pd.DataFrame(pair_block_rows)
pair_confusion_summary = pd.DataFrame(pair_confusion_rows)
pair_sample_diagnostics = pd.DataFrame(pair_sample_rows)

pair_feature_separation.to_csv(
    TABLES_DIR / "Table_predefined_pair_feature_separation.csv",
    index=False,
    encoding="utf-8-sig",
)

pair_block_separation.to_csv(
    TABLES_DIR / "Table_predefined_pair_block_separation.csv",
    index=False,
    encoding="utf-8-sig",
)

pair_confusion_summary.to_csv(
    TABLES_DIR / "Table_predefined_pair_confusion_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

pair_sample_diagnostics.to_csv(
    RESULTS_DIR / "04_predefined_pair_sample_diagnostics.csv",
    index=False,
    encoding="utf-8-sig",
)

display(pair_block_separation)
display(
    pair_feature_separation.sort_values(
        ["dataset_id", "univariate_auc_direction_adjusted"],
        ascending=[True, False],
    ).groupby("dataset_id").head(8)
)

In [ ]:
# ============================================================
# 11. Evidencia de ablación entre representaciones
# ============================================================
representation_contrasts = representation_contrasts_all.loc[
    (
        representation_contrasts_all["algorithm"] == "rbf_svm"
    )
    & (
        representation_contrasts_all["metric"] == "macro_f1"
    )
    & (
        representation_contrasts_all["contrast"].isin(
            ["R1_minus_R3", "R1_minus_R4", "R3_minus_R4"]
        )
    )
].copy()

representation_performance = model_intervals_all.loc[
    (
        model_intervals_all["algorithm"] == "rbf_svm"
    )
    & (
        model_intervals_all["probability_status"] == "native"
    )
    & (
        model_intervals_all["metric"] == "macro_f1"
    )
].copy()

representation_contrasts.to_csv(
    TABLES_DIR / "Table_rbf_svm_representation_macro_f1_contrasts.csv",
    index=False,
    encoding="utf-8-sig",
)

representation_performance.to_csv(
    TABLES_DIR / "Table_rbf_svm_representation_macro_f1_intervals.csv",
    index=False,
    encoding="utf-8-sig",
)

display(representation_contrasts)
display(representation_performance)

In [ ]:
# ============================================================
# 12. Figuras
# ============================================================
for dataset_id in DATASETS:
    # Importancia individual: orden fijado con calibración.
    calibration_data = (
        individual_importance.loc[
            (
                individual_importance["dataset_id"] == dataset_id
            )
            & (
                individual_importance["partition"] == "calibration"
            )
        ]
        .sort_values("importance_mean", ascending=True)
    )

    feature_order = calibration_data["feature"].tolist()

    figure, axis = plt.subplots(figsize=(9, 7))

    for partition in ["calibration", "test"]:
        subset = (
            individual_importance.loc[
                (
                    individual_importance["dataset_id"] == dataset_id
                )
                & (
                    individual_importance["partition"] == partition
                )
            ]
            .set_index("feature")
            .loc[feature_order]
            .reset_index()
        )

        axis.plot(
            subset["importance_mean"],
            subset["feature"],
            marker="o",
            label=partition,
        )

    axis.axvline(0.0, linewidth=1)
    axis.set_xlabel("Disminución media de macro-F1")
    axis.set_ylabel("Variable")
    axis.set_title(
        f"Importancia por permutación — {dataset_id}"
    )
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f"individual_permutation_importance_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES_DIR / f"individual_permutation_importance_{dataset_id}.pdf",
        bbox_inches="tight",
    )
    plt.show()

    # Importancia por bloques.
    block_plot = block_importance.loc[
        block_importance["dataset_id"] == dataset_id
    ].copy()

    figure, axis = plt.subplots(figsize=(8, 5))

    for partition in ["calibration", "test"]:
        subset = block_plot.loc[
            block_plot["partition"] == partition
        ].sort_values("block")

        axis.plot(
            subset["block"],
            subset["importance_mean"],
            marker="o",
            label=partition,
        )

    axis.axhline(0.0, linewidth=1)
    axis.set_ylabel("Disminución media de macro-F1")
    axis.set_xlabel("Bloque morfométrico")
    axis.set_title(
        f"Importancia conjunta de bloques — {dataset_id}"
    )
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f"block_permutation_importance_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES_DIR / f"block_permutation_importance_{dataset_id}.pdf",
        bbox_inches="tight",
    )
    plt.show()

    # Separación de los pares predefinidos.
    pair_plot = (
        pair_feature_separation.loc[
            pair_feature_separation["dataset_id"] == dataset_id
        ]
        .sort_values(
            "absolute_standardized_mean_difference",
            ascending=True,
        )
    )

    figure, axis = plt.subplots(figsize=(9, 7))
    axis.barh(
        pair_plot["feature"],
        pair_plot["standardized_mean_difference_a_minus_b"],
    )
    axis.axvline(0.0, linewidth=1)
    axis.set_xlabel(
        "Diferencia estandarizada: clase A − clase B"
    )
    axis.set_ylabel("Variable")
    axis.set_title(
        f"Separación morfométrica del par — {dataset_id}"
    )
    axis.grid(True, axis="x", alpha=0.3)
    figure.tight_layout()

    figure.savefig(
        FIGURES_DIR / f"predefined_pair_feature_separation_{dataset_id}.png",
        dpi=300,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURES_DIR / f"predefined_pair_feature_separation_{dataset_id}.pdf",
        bbox_inches="tight",
    )
    plt.show()

In [ ]:
# ============================================================
# 13. Resumen automático
# ============================================================
summary_rows = []

for dataset_id in DATASETS:
    test_blocks = block_importance.loc[
        (
            block_importance["dataset_id"] == dataset_id
        )
        & (
            block_importance["partition"] == "test"
        )
    ].sort_values("importance_mean", ascending=False)

    top_test_features = (
        individual_importance.loc[
            (
                individual_importance["dataset_id"] == dataset_id
            )
            & (
                individual_importance["partition"] == "test"
            )
        ]
        .sort_values("importance_mean", ascending=False)
        .head(5)
    )

    for _, row in test_blocks.iterrows():
        summary_rows.append(
            {
                "dataset_id": dataset_id,
                "analysis": "block_permutation_test",
                "item": row["block"],
                "estimate": row["importance_mean"],
                "interval_low": row["interval_low"],
                "interval_high": row["interval_high"],
            }
        )

    for _, row in top_test_features.iterrows():
        summary_rows.append(
            {
                "dataset_id": dataset_id,
                "analysis": "top_individual_feature_test",
                "item": row["feature"],
                "estimate": row["importance_mean"],
                "interval_low": row["interval_low"],
                "interval_high": row["interval_high"],
            }
        )

automatic_summary = pd.DataFrame(summary_rows)

automatic_summary.to_csv(
    RESULTS_DIR / "05_automatic_interpretability_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

display(automatic_summary)

In [ ]:
# ============================================================
# 14. Protocolo, validaciones y manifiesto
# ============================================================
protocol = {
    "protocol_name": (
        "Grain_AI_MDPI_ablation_interpretability_protocol_v1"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "n_permutation_repeats": N_PERMUTATION_REPEATS,
    "permutation_interval": PERMUTATION_INTERVAL,
    "primary_unit": "duplicate_group_id",
    "importance_metric": "macro_f1_drop",
    "feature_blocks": FEATURE_BLOCKS,
    "predefined_class_pairs": PAIR_DEFINITIONS,
    "model_fidelity_tolerance": 1e-10,
    "representation_inference_source": (
        "Notebook 06 paired inference; no recomputation"
    ),
    "restrictions": [
        "No model retraining.",
        "No hyperparameter reselection.",
        "No new champion selected.",
        "Feature importance is model-dependent and non-causal.",
        "Permutation intervals quantify permutation variability, not sampling confidence.",
        "Median masking is a descriptive off-manifold sensitivity analysis.",
        "Test interpretation does not alter the model or thresholds.",
        "No image-level robustness claim.",
    ],
}

(PROTOCOL_DIR / "ablation_interpretability_protocol_v1.json").write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

validation_checks = {
    "notebook06_all_checks_passed": bool(
        notebook06_validation["passed"].astype(bool).all()
    ),
    "notebook08_v2_all_checks_passed": bool(
        notebook08_validation["passed"].astype(bool).all()
    ),
    "exactly_two_frozen_champions": (
        len(training_champions) == len(DATASETS)
    ),
    "all_saved_models_reproduce_notebook05": bool(
        model_fidelity["passed"].all()
    ),
    "all_group_units_have_one_label": bool(
        (group_audit["groups_multiple_labels"] == 0).all()
    ),
    "all_group_units_have_one_feature_vector": bool(
        (group_audit["groups_multiple_vectors"] == 0).all()
    ),
    "feature_blocks_cover_all_model_features": all(
        set(
            feature
            for block_features in FEATURE_BLOCKS[dataset_id].values()
            for feature in block_features
        )
        == set(bundles[dataset_id]["feature_columns"])
        for dataset_id in DATASETS
    ),
    "individual_importance_complete": (
        len(individual_importance)
        == sum(
            len(bundles[dataset_id]["feature_columns"])
            * len(PARTITIONS_FOR_IMPORTANCE)
            for dataset_id in DATASETS
        )
    ),
    "block_importance_complete": (
        len(block_importance)
        == sum(
            len(FEATURE_BLOCKS[dataset_id])
            * len(PARTITIONS_FOR_IMPORTANCE)
            for dataset_id in DATASETS
        )
    ),
    "predefined_pairs_only": (
        set(
            zip(
                pair_feature_separation["class_a"],
                pair_feature_separation["class_b"],
            )
        )
        == set(PAIR_DEFINITIONS.values())
    ),
    "representation_contrasts_loaded_from_notebook06": (
        len(representation_contrasts) == len(DATASETS) * 3
    ),
    "no_test_based_model_selection": True,
}

final_validation = pd.DataFrame(
    [
        {"check": check, "passed": passed}
        for check, passed in validation_checks.items()
    ]
)

final_validation.to_csv(
    RESULTS_DIR / "06_final_interpretability_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

if not final_validation["passed"].all():
    display(final_validation)
    raise RuntimeError(
        "Falló al menos una validación final."
    )

def sha256_file(path: Path):
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

manifest_rows = []

for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path": str(
                    path.relative_to(OUTPUT_ROOT)
                ),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

integrity_manifest = pd.DataFrame(manifest_rows)

integrity_manifest.to_csv(
    RESULTS_DIR / "07_output_integrity_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Validación final: CORRECTA")
display(final_validation)
print("Archivos registrados:", len(integrity_manifest))

In [ ]:
# ============================================================
# 15. Exportación a Drive y ZIP
# ============================================================
DRIVE_NOTEBOOK09_ROOT = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK09_ABLATION_INTERPRETABILITY_OUTPUTS"
)

if DRIVE_NOTEBOOK09_ROOT.exists():
    shutil.rmtree(DRIVE_NOTEBOOK09_ROOT)

shutil.copytree(
    OUTPUT_ROOT,
    DRIVE_NOTEBOOK09_ROOT,
)

FINAL_ZIP_PATH = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK09_ABLATION_INTERPRETABILITY_OUTPUTS.zip"
)

if FINAL_ZIP_PATH.exists():
    FINAL_ZIP_PATH.unlink()

with zipfile.ZipFile(
    FINAL_ZIP_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=str(path.relative_to(OUTPUT_ROOT)),
            )

print("Directorio exportado:", DRIVE_NOTEBOOK09_ROOT)
print("ZIP final:", FINAL_ZIP_PATH)
print(
    "Tamaño ZIP (MB):",
    round(FINAL_ZIP_PATH.stat().st_size / 1e6, 2),
)

# Interpretación editorial

- La evidencia principal sobre el valor de tamaño frente a forma procede de los contrastes pareados de representaciones del Notebook 06.
- La importancia por permutación explica la dependencia del modelo congelado dentro de cada representación.
- Las variables correlacionadas pueden repartirse o sustituirse entre sí; una importancia individual pequeña no implica ausencia de información.
- La permutación conjunta de bloques es más apropiada para la pregunta tamaño–forma.
- La estabilidad entre calibración y prueba debe considerarse antes de destacar una variable.
- El enmascaramiento con medianas es un análisis de sensibilidad y no debe describirse como una intervención causal.
- Los pares de clases fueron preespecificados y no seleccionados mediante este análisis.